# RSNA Knee — fine-tuned ResNet18 (3 planes, attention pooling over slices)
GPU必須。Internet ONでImageNet重みを取得。画像は224px×16スライスをディスクにキャッシュし、エンドツーエンドで学習する。

In [ ]:
import os, sys, subprocess, warnings
os.environ['PYTHONWARNINGS'] = 'ignore'  # joblibのワーカープロセスにも効かせる
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from pathlib import Path
from joblib import Parallel, delayed
_name = 'rsna-knee-abnormality-detection'
_cands = [Path('/kaggle/input/competitions')/_name, Path('/kaggle/input')/_name]
COMP = next((d for d in _cands if (d/'train.csv').exists()), None)
assert COMP, f"train.csv が見つからない。/kaggle/input: {[p.name for p in Path('/kaggle/input').iterdir()]}"
print('COMP =', COMP)
SRC = Path('/kaggle/working/repo/rsna-knee-abnormality-detection/src')
if not SRC.exists():  # internet ON のときだけ。OFFなら src を Dataset で追加して SRC を変更
    subprocess.run(['git','clone','--depth','1','-b','main',
                    'https://github.com/hidenori24/kaggle_ipynb','/kaggle/working/repo'], check=True)
sys.path.insert(0, str(SRC))
from pseudo_labels import LABELS, add_pseudo_labels
import finetune as ft, torch
print('commit:', subprocess.run(['git','-C',str(SRC),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
train = pd.read_csv(COMP/'train.csv'); tseries = pd.read_csv(COMP/'train_series.csv')
test = pd.read_csv(COMP/'test.csv');   teseries = pd.read_csv(COMP/'test_series.csv')
print(train.shape, tseries.shape, test.shape)
print(train[LABELS].notna().sum())   # 正解ラベルがあるstudy数
train['has_true'] = train['ACL'].notna()   # 正解ラベルあり(疑似ラベルではない)
train = add_pseudo_labels(train)
# 英語以外のレポートは疑似ラベルが全て0になり「誤った陰性」になるので、疑似ラベル学習から外す
def ascii_share(t):
    t = str(t); return sum(ch.isascii() for ch in t) / max(len(t), 1)
train['is_en'] = train['Report'].map(ascii_share) > 0.97
print('english-like reports:', train.is_en.sum(), '/', len(train))
print('pseudo-positive rate  en vs non-en:'); print(train.groupby('is_en')[LABELS].mean().round(3).T)
# 非英語レポートの言語推定用の集計(本文は出力しない): 文字種の内訳と頻出語
import unicodedata, re
from collections import Counter
non = train.loc[~train.is_en, 'Report'].fillna('').astype(str)
def script(t):
    c = Counter()
    for ch in t:
        if ch.isalpha():
            c[unicodedata.name(ch, 'X').split()[0]] += 1
    return c.most_common(1)[0][0] if c else 'NONE'
print('non-English dominant script:', Counter(non.map(script)).most_common(6))
words = Counter(w for t in non for w in set(re.findall(r'[^\W\d_]{4,}', t.lower())))
print('top words (doc freq):', words.most_common(60))
print(train[LABELS].mean().round(3))

In [ ]:
SIZE, K, EPOCHS, BS = 224, 16, 14, 8
CACHE = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')   # /kaggle/working に置くと出力に含まれてしまう
use = train[train.has_true | train.is_en].reset_index(drop=True)     # 英語レポートの疑似ラベル + 正解ラベル付き
print('studies:', len(use), '| true-labeled:', int(use.has_true.sum()))
X, M = ft.cache_volumes(use, tseries, COMP/'train_series', SIZE, K, CACHE, 'train')
print('planes available (Sag/Cor/Ax):', M.sum(0))
Y = use[LABELS].values.astype(np.float32)

In [ ]:
rng = np.random.RandomState(0)
true_idx = np.where(use.has_true.values)[0]                       # 58件は評価専用(正解ラベル)
pseudo = np.where(~use.has_true.values)[0]; rng.shuffle(pseudo)
n_va = int(0.1 * len(pseudo))
va_pseudo, tr_idx = pseudo[:n_va], pseudo[n_va:]
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev, '| train', len(tr_idx), 'val-pseudo', len(va_pseudo))
model, hist = ft.fit(X, M, Y, tr_idx, {'pseudo': va_pseudo, 'true': true_idx}, dev, epochs=EPOCHS, bs=BS)

In [ ]:
print('best:', max(hist, key=lambda h: h.get('pseudo', -1)))
OUT = Path('model_out'); OUT.mkdir(exist_ok=True)
torch.save({'state': model.state_dict(), 'size': SIZE, 'k': K, 'labels': LABELS}, OUT/'ft_model.pt')